# Phase 0 — verify the record

VIABILITY_PLAN.md §4: confirm the documented facts hold on disk before testing claims about them.
One cell per item, documented vs measured, provenance inline. Run All; the summary cell at the
bottom prints every check as PASS / MISMATCH. Numbers cited in VERDICT.md reference these cells.

In [1]:

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from hybrid_search_rrf_dataset.router import (
    DATA_DIR,
    AutoFusionRouter,
    Representation,
    RouterExperiment,
    decisive_rows,
)

ROUTES = ["dense_only", "pure_rrf", "sparse_only"]
SCORE_COLS = [f"score_{r}" for r in ROUTES]
checks: list[dict] = []  # the summary cell reads this


def check(name: str, documented, measured, tol: float = 0.0) -> None:
    ok = (
        abs(measured - documented) <= tol
        if isinstance(documented, (int, float)) and tol
        else measured == documented
    )
    checks.append({"check": name, "documented": documented, "measured": measured, "ok": ok})
    print(f"{'PASS    ' if ok else 'MISMATCH'} {name}: documented={documented} measured={measured}")

/Users/andrei/projects/hybrid-search-rrf-dataset/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# item 1 — label counts (documented: 24,338 labelled / 2,510 decisive / dense 1,858 / sparse 507 / rrf 145)
labels = pd.read_parquet(DATA_DIR / "route_labels" / "labels.parquet")
print(f"labels.parquet: {len(labels):,} total rows")

exp = RouterExperiment()
data = exp.load()  # labels ⋈ catalog (inner) — the frame every experiment runs on
check("working frame rows (labels ⋈ catalog)", 24338, len(data))

dec = decisive_rows(data)
check("decisive rows", 2510, len(dec))
split = dec["winner"].value_counts()
for route, documented in [("dense_only", 1858), ("sparse_only", 507), ("pure_rrf", 145)]:
    check(f"decisive winner {route}", documented, int(split.get(route, 0)))
print("\nshape value counts (working frame):")
print(data["shape"].value_counts())

labels.parquet: 46,142 total rows
MISMATCH working frame rows (labels ⋈ catalog): documented=24338 measured=46142
MISMATCH decisive rows: documented=2510 measured=5100
MISMATCH decisive winner dense_only: documented=1858 measured=3149
MISMATCH decisive winner sparse_only: documented=507 measured=1731
MISMATCH decisive winner pure_rrf: documented=145 measured=220

shape value counts (working frame):
shape
routes_differ    22943
all_tied         15018
all_zero          8181
Name: count, dtype: int64


In [3]:
# item 2 — ceilings over ALL working-frame rows (d44(a): always-dense 0.481 / best-constant-per-collection 0.511 / oracle 0.557)
scores = data[SCORE_COLS].to_numpy()
always_dense = float(data["score_dense_only"].mean())

per_lane_best = data.groupby("dataset")[SCORE_COLS].mean().idxmax(axis=1)  # best route per collection
best_const = float(
    sum(data.loc[data["dataset"] == lane, col].sum() for lane, col in per_lane_best.items()) / len(data)
)
oracle = float(scores.max(axis=1).mean())

check("always-dense", 0.481, round(always_dense, 3), tol=0.002)
check("best constant per collection", 0.511, round(best_const, 3), tol=0.002)
check("per-query oracle", 0.557, round(oracle, 3), tol=0.002)
print(f"\nheadroom (oracle − best const): {oracle - best_const:.3f}")

MISMATCH always-dense: documented=0.481 measured=0.565
MISMATCH best constant per collection: documented=0.511 measured=0.602
MISMATCH per-query oracle: documented=0.557 measured=0.66

headroom (oracle − best const): 0.057


In [4]:
# item 3 — lane inventory: corpus.parquet presence + doc counts; oracle dirs; documented "16 lanes" vs disk
lanes = []
for d in sorted(DATA_DIR.iterdir()):
    if d.is_dir() and (d / "corpus.parquet").exists():
        lanes.append({"lane": d.name, "docs": pq.read_metadata(d / "corpus.parquet").num_rows})
lanes = pd.DataFrame(lanes)
oracle_dirs = sorted((DATA_DIR / "route_labels").glob("*_oracle"))
oracle_rows = sum(pq.read_metadata(d / "rows.parquet").num_rows for d in oracle_dirs if (d / "rows.parquet").exists())

print(f"lanes with corpus.parquet: {len(lanes)} (documented: 16)")
print(f"oracle dirs: {len(oracle_dirs)}, total oracle rows: {oracle_rows:,} (labels.parquet: {len(labels):,})")
check("oracle rows == labels rows", len(labels), oracle_rows)
lanes.sort_values("docs", ascending=False)

lanes with corpus.parquet: 43 (documented: 16)
oracle dirs: 42, total oracle rows: 46,142 (labels.parquet: 46,142)
PASS     oracle rows == labels rows: documented=46142 measured=46142


,lane,docs
16,crumb-code-retrieval,119976
15,crumb-clinical-trial,100000
37,rarb-code,100000
34,msmarco-passage-dev,100000
31,lotte-technology-forum,79450
23,dbpedia-entity,74385
36,quest,72080
35,orcas,62250
19,crumb-set-operation-entity-retrieval,57730
30,limit,50000


In [5]:
# item 4 — reproduce the record. Two vintages in circulation:
#   PLAN.md: 0.751 router / 0.738 comparator
#   route_experiments.ipynb validate cell: 0.693 router / 0.625 const_dense (decisive, random_within_lane)
# Config matches the notebook vintage: ENGINEERED, all_rows='decisive'.
result = exp.run(representations=[Representation.ENGINEERED], all_rows="decisive")
cols = ["protocol", "n_test_decisive", "const_dense_only", "const_pure_rrf",
        "const_sparse_only", "oracle", "router", "headroom_captured"]
rw = result[result["protocol"] == "random_within_lane"].iloc[0]
check("record: router (decisive, random split)", 0.693, round(float(rw["router"]), 3), tol=0.005)
check("record: const dense (decisive, random split)", 0.625, round(float(rw["const_dense_only"]), 3), tol=0.005)
result[cols].round(3)

router ablation:   0%|          | 0/2 [00:00<?, ?it/s]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, fitting]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, tuning thresholds]

random_within_lane·engineered:   0%|          | 0/2 [00:00<?, ?it/s, evaluating]       

...

holdout_lane·engineered:  50%|█████     | 1/2 [00:00<00:00,  1.49it/s, headroom=0.153, n=206]

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00,  2.59it/s, headroom=0.153, n=206]

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00,  2.33it/s, headroom=0.153, n=206]

MISMATCH record: router (decisive, random split): documented=0.693 measured=0.712
MISMATCH record: const dense (decisive, random split): documented=0.625 measured=0.631


,protocol,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured
0,random_within_lane,1021,0.631,0.208,0.367,0.975,0.712,0.237
1,holdout_lane,206,0.607,0.185,0.430,1.000,0.667,0.153


In [6]:
# item 5 — the two known defects, quantified so no later number silently contains them
# (a) round-trip inconsistency (~147 of 46K documented): served strategy's `metric`
#     should equal route_scores[strategy_name]
mismatch_rows = 0
nf_rows = None
for d in oracle_dirs:
    f = d / "rows.parquet"
    if not f.exists():
        continue
    rows = pd.read_parquet(f, columns=["strategy_name", "metric", "route_scores"])
    served = rows.apply(lambda r: r["route_scores"].get(r["strategy_name"]), axis=1)
    mismatch_rows += int((~np.isclose(served.astype(float), rows["metric"].astype(float), atol=1e-6)).sum())
    if d.name == "beir-nfcorpus_oracle":
        nf_rows = len(rows)
print(f"(a) served-metric vs route_scores mismatches: {mismatch_rows} of {oracle_rows:,} (documented ~147)")
print(f"(b) beir-nfcorpus_oracle rows on disk: {nf_rows} (documented stale cache: 323 rows — fewer means it was cleaned)")
checks.append({"check": "round-trip mismatches", "documented": "~147", "measured": mismatch_rows, "ok": True})

(a) served-metric vs route_scores mismatches: 0 of 46,142 (documented ~147)
(b) beir-nfcorpus_oracle rows on disk: 12 (documented stale cache: 323 rows — fewer means it was cleaned)


In [7]:
# item 6 — qrel holes per route: share of each route's stored top-10 with NO judgment at all.
# Literature: dense 14-32% vs BM25 ~6% (Hole@10). A large dense-minus-sparse gap means the
# label pool is biased toward whichever route fed the judgments.
hole_rows = []
for d in oracle_dirs:
    lane = d.name.removesuffix("_oracle")
    qrels_path = DATA_DIR / lane / "qrels.parquet"
    f = d / "rows.parquet"
    if not f.exists() or not qrels_path.exists():
        print(f"skip {lane}: missing rows or qrels")
        continue
    qrels = pd.read_parquet(qrels_path, columns=["query_id", "doc_id"])
    judged = set(zip(qrels["query_id"].astype(str), qrels["doc_id"].astype(str)))
    rows = pd.read_parquet(f, columns=["query_id", "route_rankings"])
    for route in ROUTES:
        total = holes = 0
        for qid, rankings in zip(rows["query_id"].astype(str), rows["route_rankings"]):
            for doc in rankings[route]:
                total += 1
                holes += (qid, str(doc)) not in judged
        if total:
            hole_rows.append({"lane": lane, "route": route, "docs": total, "holes": holes,
                              "hole_rate": holes / total})
holes_df = pd.DataFrame(hole_rows)
pooled = holes_df.groupby("route").apply(lambda g: g["holes"].sum() / g["docs"].sum(), include_groups=False)
print("pooled hole rate per route:")
print(pooled.round(3))
gap = float(pooled["dense_only"] - pooled["sparse_only"])
print(f"dense-minus-sparse gap: {gap:+.3f} (literature reference: dense 14-32% vs BM25 ~6%)")
checks.append({"check": "qrel hole gap (dense-sparse)", "documented": "unmeasured", "measured": round(gap, 3), "ok": True})
holes_df.pivot(index="lane", columns="route", values="hole_rate").round(3)

pooled hole rate per route:
route
dense_only     0.891
pure_rrf       0.882
sparse_only    0.897
dtype: float64
dense-minus-sparse gap: -0.007 (literature reference: dense 14-32% vs BM25 ~6%)


route,dense_only,pure_rrf,sparse_only
lane,,,
antique,0.264,0.154,0.139
beir-nfcorpus,0.750,0.717,0.730
bright-aops,1.000,1.000,1.000
bright-biology,0.894,0.860,0.873
bright-earth-science,0.838,0.803,0.822
bright-economics,0.853,0.863,0.903
bright-leetcode,0.850,0.800,0.850
bright-pony,0.957,0.890,0.867
bright-psychology,0.876,0.877,0.908


In [8]:
# item 7 — auto-fusion cache coverage: prices the three gates before Phase 1 commits to them
cache_path = AutoFusionRouter.CACHE_PATH
if cache_path.exists():
    cache = pd.read_parquet(cache_path)
    covered = len(cache.merge(data[["dataset", "query_id"]].drop_duplicates(), on=["dataset", "query_id"]))
else:
    covered = 0
print(f"cache path: {cache_path}")
print(f"exists: {cache_path.exists()} — coverage {covered:,} of {len(data):,} working-frame rows")
print("every uncached row costs one HTTP classifier call when the gates run")
checks.append({"check": "autofusion cache coverage", "documented": "unknown", "measured": f"{covered}/{len(data)}", "ok": True})

cache path: /Users/andrei/projects/hybrid-search-rrf-dataset/src/data/route_labels/autofusion_cache.parquet
exists: False — coverage 0 of 46,142 working-frame rows
every uncached row costs one HTTP classifier call when the gates run


In [9]:
# item 8 — raw per-strategy score persistence (decides Phase 2's scope).
# The plan's premise was that raw scores were never written (golden.py:66 vintage).
sample = pd.read_parquet(oracle_dirs[0] / "rows.parquet", columns=["route_raw_scores", "route_rankings"])
depths = pd.DataFrame([
    {r: len(row["route_raw_scores"][r]) for r in ROUTES} for _, row in sample.iterrows()
])
print("route_raw_scores EXISTS in oracle rows.parquet (all 42 dirs share the schema).")
print(f"depth distribution in {oracle_dirs[0].name}: min={depths.min().min()}, max={depths.max().max()}")
print("\nConsequence for Phase 2: raw scores exist at TOP-10 ONLY.")
print("  - offline OK: qrel holes, Gate 3 slice, probe tables, C1/C1b replay (stored objectives)")
print("  - still needs the depth-50 relog: exact RRF-k sweep, DBSF, weighted fusion (a doc at")
print("    dense rank 23 + sparse rank 4 can enter a fused top-10; rank 11+ is not on disk)")
checks.append({"check": "raw scores persisted", "documented": "never written (stale)", "measured": "top-10 per route", "ok": True})

route_raw_scores EXISTS in oracle rows.parquet (all 42 dirs share the schema).
depth distribution in antique_oracle: min=0, max=10

Consequence for Phase 2: raw scores exist at TOP-10 ONLY.
  - offline OK: qrel holes, Gate 3 slice, probe tables, C1/C1b replay (stored objectives)
  - still needs the depth-50 relog: exact RRF-k sweep, DBSF, weighted fusion (a doc at
    dense rank 23 + sparse rank 4 can enter a fused top-10; rank 11+ is not on disk)


In [10]:
# summary — every check, PASS / MISMATCH
summary = pd.DataFrame(checks)
n_bad = int((~summary["ok"]).sum())
print(f"{len(summary)} checks, {n_bad} mismatches")
summary

15 checks, 10 mismatches


,check,documented,measured,ok
0,working frame rows (labels ⋈ catalog),24338,46142,False
1,decisive rows,2510,5100,False
2,decisive winner dense_only,1858,3149,False
3,decisive winner sparse_only,507,1731,False
4,decisive winner pure_rrf,145,220,False
5,always-dense,0.481,0.565,False
6,best constant per collection,0.511,0.602,False
7,per-query oracle,0.557,0.66,False
8,oracle rows == labels rows,46142,46142,True
9,"record: router (decisive, random split)",0.693,0.712,False
